In [ ]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

train_df = pd.read_csv(RAW_DATA_DIR / "train.csv")
validation_df = pd.read_csv(RAW_DATA_DIR / "validation.csv")

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)

Train shape: (4000, 8)
Validation shape: (800, 8)


In [2]:
languages = train_df["language"].unique()

for language in languages:
    print(f"\n{'=' * 60}")
    print(f"Language: {language}")
    print("=" * 60)
    
    samples = train_df[train_df["language"] == language].sample(
        3, random_state=42
    )
    
    for _, row in samples.iterrows():
        print(f"\nCategory: {row['category']}")
        print(f"Text: {row['text']}")


Language: en

Category: payment_refund
Text: um um agent:  thank you for calling rideeat

how may i
assist customer: the    screen shows the ref simb-h6t3smz agent: pls

explain the issue
customer:
i have been trying
to  sort this out i do not remember  the  exact minute i

opened the    service
but  i    have  the    ref i usually  keep the booking record  until  everything has been sorted out the return of my  money was approved but nothing
has    reached my wallet pls respond when someone has checked  this agent: thank you  i  have noted    the request

Category: ride_trip_issue
Text: i have  a    problem
w/
recent
bokking
the car went to  the  wrong  pickup entrance and  then left i    selected    service  earlier today can you help me
resolve    this 🙏

Category: delivery_delay
Text: this relates to booking
under  simq-6rysjqs i asked  family member  to  help me write msg  i  was
near
dehiwala
when
this
happened  hi  i need help
with something  my oder still  at preparing
stage a

In [3]:
for language in train_df["language"].unique():
    print(f"\n{'=' * 50}")
    print(f"LANGUAGE: {language}")
    print("=" * 50)

    samples = train_df[train_df["language"] == language].sample(
        2, random_state=42
    )

    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"\nSample {i}")
        print(f"Category: {row['category']}")
        print(f"Text: {row['text'][:500]}")


LANGUAGE: en

Sample 1
Category: payment_refund
Text: um um agent:  thank you for calling rideeat

how may i
assist customer: the    screen shows the ref simb-h6t3smz agent: pls

explain the issue
customer:
i have been trying
to  sort this out i do not remember  the  exact minute i

opened the    service
but  i    have  the    ref i usually  keep the booking record  until  everything has been sorted out the return of my  money was approved but nothing
has    reached my wallet pls respond when someone has checked  this agent: thank you  i  have note

Sample 2
Category: ride_trip_issue
Text: i have  a    problem
w/
recent
bokking
the car went to  the  wrong  pickup entrance and  then left i    selected    service  earlier today can you help me
resolve    this 🙏

LANGUAGE: singlish

Sample 1
Category: payment_refund
Text: mokakda mekata karanna ona? pennana reference eka simb-n49ur47. mama phone eken baluwe. mama ayeth details check kala. mage awasarayak nathuwa card eken dan salli kapen

In [5]:
train_df["text_length"] = train_df["text"].str.len()
train_df["word_count"] = train_df["text"].str.split().str.len()

language_stats = (
    train_df.groupby("language")
    .agg(
        tickets=("text", "size"),
        avg_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        avg_chars=("text_length", "mean"),
        median_chars=("text_length", "median"),
    )
    .round(2)
)

language_stats

,tickets,avg_words,median_words,avg_chars,median_chars
language,,,,,
en,1400,72.61,73.5,414.22,427.5
mixed,100,43.72,42.5,291.59,265.5
si,800,38.11,35.0,241.91,218.0
singlish,1000,39.40,38.0,260.62,252.0
ta,600,32.89,31.0,303.42,279.0
tanglish,100,36.01,33.5,266.95,238.5


In [6]:
from collections import Counter

# Inspect the most frequent raw words in each language
for language in train_df["language"].unique():
    texts = train_df.loc[train_df["language"] == language, "text"]
    words = " ".join(texts).lower().split()
    
    print(f"\n{language.upper()}")
    print(Counter(words).most_common(20))


EN
[('the', 8423), ('i', 7347), ('to', 2377), ('this', 2178), ('have', 1770), ('a', 1551), ('you', 1480), ('me', 1362), ('is', 1207), ('can', 1104), ('agent:', 1006), ('for', 927), ('please', 922), ('and', 850), ('my', 806), ('rideeat', 765), ('need', 747), ('was', 742), ('thank', 738), ('help', 733)]

SINGLISH
[('eka', 1820), ('details', 1197), ('mama', 971), ('ekak', 781), ('me', 749), ('thawa', 600), ('mokakda', 586), ('eke', 583), ('meka', 577), ('customer:', 522), ('help', 479), ('agent:', 472), ('gana', 467), ('wela', 440), ('kiyanna', 440), ('message', 405), ('phone', 385), ('na.', 384), ('ayeth', 373), ('tikak', 361)]

MIXED
[('checked', 139), ('the', 139), ('please', 121), ('i', 113), ('this', 110), ('reply', 97), ('when', 97), ('you', 97), ('have', 97), ('විස්තර', 63), ('details', 62), ('this.', 60), ('on', 43), ('my', 43), ('check', 43), ('and', 43), ('tell', 42), ('me', 42), ('next', 42), ('is', 41)]

TA
[('நான்', 488), ('மேலும்', 375), ('சொல்லுங்கள்.', 346), ('விவரங்களை',

In [7]:
# Measure the proportion of Latin and non-Latin characters by language
def character_profile(text):
    chars = [c for c in text if not c.isspace()]
    latin = sum(c.isascii() and c.isalpha() for c in chars)
    non_latin = sum(c.isalpha() and not c.isascii() for c in chars)
    return latin, non_latin

for language in train_df["language"].unique():
    counts = train_df.loc[
        train_df["language"] == language, "text"
    ].apply(character_profile)

    latin_chars = sum(x[0] for x in counts)
    non_latin_chars = sum(x[1] for x in counts)
    total_letters = latin_chars + non_latin_chars

    print(
        f"{language.upper():10} "
        f"Latin: {latin_chars / total_letters:.2%} | "
        f"Non-Latin: {non_latin_chars / total_letters:.2%}"
    )

EN         Latin: 100.00% | Non-Latin: 0.00%
SINGLISH   Latin: 100.00% | Non-Latin: 0.00%
MIXED      Latin: 53.62% | Non-Latin: 46.38%
TA         Latin: 3.76% | Non-Latin: 96.24%
SI         Latin: 4.60% | Non-Latin: 95.40%
TANGLISH   Latin: 100.00% | Non-Latin: 0.00%


In [8]:
# Check how often speaker labels and reference IDs appear in tickets
patterns = {
    "agent_label": r"\bagent:",
    "customer_label": r"\bcustomer:",
    "reference_id": r"\bsimb-[a-z0-9]+\b",
}

for name, pattern in patterns.items():
    matches = train_df["text"].str.contains(pattern, case=False, regex=True)
    print(f"{name}: {matches.sum()} ({matches.mean():.2%})")

agent_label: 637 (15.93%)
customer_label: 637 (15.93%)
reference_id: 272 (6.80%)


In [9]:
# Compare structural text patterns across primary categories
pattern_stats = train_df.groupby("category").agg(
    tickets=("ticket_id", "size"),
    agent_label=("text", lambda x: x.str.contains(r"\bagent:", case=False, regex=True).mean()),
    customer_label=("text", lambda x: x.str.contains(r"\bcustomer:", case=False, regex=True).mean()),
    reference_id=("text", lambda x: x.str.contains(r"\bsimb-[a-z0-9]+\b", case=False, regex=True).mean()),
)

pattern_stats.round(3)

,tickets,agent_label,customer_label,reference_id
category,,,,
account_promo,360,0.161,0.161,0.067
app_technical,320,0.134,0.134,0.044
delivery_delay,600,0.163,0.163,0.080
food_quality,280,0.164,0.164,0.075
general_inquiry,200,0.160,0.160,0.065
lost_item,200,0.150,0.150,0.035
order_missing_wrong,480,0.162,0.162,0.069
payment_refund,720,0.181,0.181,0.082
ride_trip_issue,560,0.166,0.166,0.075


In [10]:
# Inspect examples of reference IDs found in the ticket text
reference_samples = train_df.loc[
    train_df["text"].str.contains(r"\bsimb-[a-z0-9]+\b", case=False, regex=True),
    "text"
].str.extract(r"(\bsimb-[a-z0-9]+\b)", expand=False)

reference_samples.head(20).tolist()

['simb-4pvsfpq',
 nan,
 'simb-gk5c6rl',
 'simb-p6lkdmv',
 'simb-8qquqkr',
 nan,
 nan,
 'simb-e76vkvy',
 'simb-k73ge5m',
 nan,
 nan,
 nan,
 nan,
 nan,
 'simb-9pxwgme',
 'simb-lfllr6x',
 nan,
 'simb-estafwq',
 nan,
 nan]

In [11]:
from collections import Counter

# Inspect the most frequent raw words in each primary category
for category in train_df["category"].unique():
    texts = train_df.loc[train_df["category"] == category, "text"]
    words = " ".join(texts).lower().split()

    print(f"\n{category.upper()}")
    print(Counter(words).most_common(15))


PAYMENT_REFUND
[('the', 1613), ('i', 1359), ('to', 461), ('this', 435), ('me', 393), ('have', 346), ('a', 341), ('you', 338), ('agent:', 321), ('details', 309), ('eka', 276), ('rideeat', 260), ('customer:', 260), ('විස්තර', 242), ('මම', 236)]

ORDER_MISSING_WRONG
[('the', 1060), ('i', 864), ('this', 278), ('to', 257), ('eka', 236), ('me', 235), ('details', 219), ('have', 203), ('විස්තර', 199), ('මම', 198), ('you', 181), ('a', 177), ('agent:', 175), ('rideeat', 164), ('customer:', 156)]

DELIVERY_DELAY
[('the', 1288), ('i', 1064), ('this', 362), ('to', 361), ('me', 318), ('have', 295), ('eka', 293), ('details', 248), ('is', 235), ('you', 232), ('agent:', 229), ('rideeat', 216), ('help', 204), ('විස්තර', 204), ('please', 198)]

ACCOUNT_PROMO
[('the', 787), ('i', 748), ('to', 231), ('this', 213), ('eka', 211), ('me', 202), ('have', 188), ('you', 145), ('details', 140), ('agent:', 139), ('rideeat', 137), ('විස්තර', 136), ('my', 124), ('a', 121), ('please', 117)]

APP_TECHNICAL
[('the', 65

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2
)

X_tfidf = vectorizer.fit_transform(train_df["text"])

feature_names = vectorizer.get_feature_names_out()

for category in train_df["category"].unique():
    category_mask = train_df["category"] == category
    category_scores = X_tfidf[category_mask].mean(axis=0).A1
    top_indices = category_scores.argsort()[-15:][::-1]

    print(f"\n{category.upper()}")
    print([feature_names[i] for i in top_indices])


PAYMENT_REFUND
['the', 'கள', 'යන', 'this', 'to', 'card', 'එක', 'මම', 'තර', 'me', 'eka', 'details', 'agent', 'you', 'have']

ORDER_MISSING_WRONG
['the', 'යන', 'கள', 'this', 'මම', 'eka', 'තර', 'details', 'me', 'to', 'කය', 'ekak', 'agent', 'have', 'you']

DELIVERY_DELAY
['the', 'கள', 'eka', 'this', 'තර', 'යන', 'to', 'මම', 'na', 'me', 'details', 'kema', 'have', 'agent', 'is']

ACCOUNT_PROMO
['the', 'கள', 'eka', 'යන', 'this', 'to', 'තර', 'me', 'මම', 'have', 'code', 'உள', 'details', 'agent', 'my']

APP_TECHNICAL
['the', 'eka', 'கள', 'යන', 'this', 'map', 'to', 'මම', 'me', 'තර', 'details', 'app', 'යම', 'have', 'open']

RIDE_TRIP_ISSUE
['the', 'යන', 'கள', 'තර', 'this', 'මම', 'eka', 'me', 'to', 'details', 'ekak', 'agent', 'driver', 'kala', 'have']

GENERAL_INQUIRY
['the', 'கள', 'me', 'to', 'this', 'යන', 'තර', 'service', 'help', 'have', 'you', 'details', 'eka', 'මම', 'explain']

SAFETY_CONDUCT
['the', 'கள', 'driver', 'මම', 'තර', 'යද', 'යන', 'to', 'mama', 'eka', 'ekak', 'this', 'safe', 'நர', 'ஓட'

In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inspect common character patterns captured by a character TF-IDF vectorizer
char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    min_df=3
)

X_char = char_vectorizer.fit_transform(train_df["text"])
char_features = char_vectorizer.get_feature_names_out()

print("Number of character features:", len(char_features))
print("Sample character n-grams:")
print(char_features[:100])

Number of character features: 45322
Sample character n-grams:
['\n15' '\n150' '\n1500' '\n15k' '\n15k ' '\n22' '\n220' '\n2200' '\n85'
 '\n850' '\n850 ' '\n> ' '\n> e' '\n> ea' '\n> i' '\n> i ' '\n> m'
 '\n> my' '\n> p' '\n> pl' '\n> t' '\n> th' '\na ' '\na c' '\na ca'
 '\na cu' '\na f' '\na fa' '\na p' '\na pr' '\nab' '\nabo' '\nabou' '\nac'
 '\nacc' '\nacce' '\nad' '\nadd' '\naddr' '\naf' '\naft' '\nafte' '\nag'
 '\naga' '\nagai' '\nage' '\nagen' '\nah' '\naha' '\nahal' '\nal' '\nall'
 '\nall ' '\nalt' '\nalth' '\nam' '\nam ' '\nam g' '\nam t' '\nama'
 '\namar' '\namo' '\namou' '\nan' '\nan ' '\nand' '\nand\n' '\nand '
 '\nano' '\nanot' '\nap' '\napp' '\napp\n' '\napp ' '\napp.' '\nappr'
 '\nar' '\nara' '\naran' '\naro' '\narou' '\narr' '\narri' '\nas' '\nas '
 '\nas m' '\nask' '\naske' '\naski' '\nass' '\nassi' '\nat' '\nat '
 '\nat t' '\nath' '\nathu' '\naw' '\nawe' '\nawe ' '\nawi']


In [15]:
# Check whether identical ticket text appears multiple times
text_counts = train_df["text"].value_counts()

print("Unique texts:", text_counts.nunique())
print("Texts appearing more than once:", (text_counts > 1).sum())
print("Maximum repetitions:", text_counts.max())

Unique texts: 1
Texts appearing more than once: 0
Maximum repetitions: 1


In [16]:
# Check whether any ticket text appears in both train and validation
train_texts = set(train_df["text"])
validation_texts = set(validation_df["text"])

overlap = train_texts.intersection(validation_texts)

print("Train-validation text overlap:", len(overlap))

Train-validation text overlap: 0


In [17]:
import re

# Normalize reference IDs and numbers to detect repeated text templates
def normalize_template(text):
    text = re.sub(r"\bsimb-[a-z0-9]+\b", "<REF>", text, flags=re.IGNORECASE)
    text = re.sub(r"\d+", "<NUM>", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

normalized_text = train_df["text"].apply(normalize_template)

template_counts = normalized_text.value_counts()

print("Unique normalized templates:", template_counts.size)
print("Repeated templates:", (template_counts > 1).sum())
print("Maximum repetitions:", template_counts.max())

Unique normalized templates: 4000
Repeated templates: 0
Maximum repetitions: 1


In [18]:
# Inspect examples from categories that may be difficult to distinguish
category_pairs = [
    ("payment_refund", "order_missing_wrong"),
    ("delivery_delay", "order_missing_wrong"),
    ("ride_trip_issue", "safety_conduct"),
    ("app_technical", "account_promo"),
]

for category_a, category_b in category_pairs:
    print(f"\n{'=' * 70}")
    print(f"{category_a.upper()} vs {category_b.upper()}")
    print("=" * 70)

    for category in [category_a, category_b]:
        samples = train_df[train_df["category"] == category].sample(
            2, random_state=42
        )

        print(f"\n--- {category} ---")

        for i, (_, row) in enumerate(samples.iterrows(), 1):
            print(f"\nSample {i}:")
            print(row["text"][:400])


PAYMENT_REFUND vs ORDER_MISSING_WRONG

--- payment_refund ---

Sample 1:
කරුණාකර මේක බලන්න. අදාළ තිරයේ පිටපතක් මම තියාගත්තා. තහවුරු කිරීමේ විස්තර මම තියාගෙන ඉන්නවා. මම යාපනය පැත්තේ හිටියේ. ආපහු දෙනවා කිව්ව මුදල තාම ලැබිලා නැහැ. මේ ගැන හොයලා දැනුම් දෙන්න. 😕

Sample 2:
The screen shows the reference SIMM-9W4EGHH.
I saved the receipt for reference.
I have a problem with a recent booking.
The wallet top-up went through at the bank but the balance is unchanged. The driver also ignored my requested intermediate stop.
Can you help me resolve this?

--- order_missing_wrong ---

Sample 1:
mage request eka balanna.

details hoyala liyanna tikak wela giya.

kalin messages tikath balala meka liyanne.

message eka liyaddi wena wadakata yanna una.

parcel eka awath danma ganna ona behetha athule na wena dose ekak gedara na.

pennana reference eka SIMB-2VKCLXF.

mama phone eken baluwe.

mama kalinma booking eka dala thibbe.

reply ekak denna.

Sample 2:
I am writing about a service problem The ref s

In [19]:
# Inspect examples from the remaining potentially confusing category pairs
category_pairs = [
    ("delivery_delay", "order_missing_wrong"),
    ("ride_trip_issue", "safety_conduct"),
    ("app_technical", "account_promo"),
]

for category_a, category_b in category_pairs:
    print(f"\n{'=' * 70}")
    print(f"{category_a.upper()} vs {category_b.upper()}")
    print("=" * 70)

    for category in [category_a, category_b]:
        samples = train_df[train_df["category"] == category].sample(
            2, random_state=42
        )

        print(f"\n--- {category} ---")

        for i, (_, row) in enumerate(samples.iterrows(), 1):
            print(f"\nSample {i}:")
            print(row["text"][:400])


DELIVERY_DELAY vs ORDER_MISSING_WRONG

--- delivery_delay ---

Sample 1:
please meka balanna.
screen eke screenshot ekak thiyagaththa.
payment record eke LKR 1500 kiyala thiyenawa.
babage formula order eka thama na next feed ekata gedara formula ithuru na.
reply ekak denna.

Sample 2:
uh hello முகவர்:  RideEat ஐ அழைத்ததற்கு நன்றி விவரங்களைச் சொல்லுங்கள் வாடிக்கையாளர்: இதைக்    கொஞ்சம் பார்த்துச் சொல்ல முடியுமா விவரங்களைத் தேடி  எழுதச் சிறிது நேரம்

எடுத்தது    முகவர்: மேலும் விவரங்களைச் சொல்லுங்கள் வாடிக்கையாளர்: சாப்பாடு கொண்டு
வருபவர் ஒரே இடத்தில்
இருக்கிறார் அழைப்புக்கும் பதில் இல்லை என் குடும்பத்தினர் ஒருவர்

செய்தி    எழுத
உதவினார் நான் முன்பே பதிவு செய்திருந்தேன் முடியும்

--- order_missing_wrong ---

Sample 1:
mage request eka balanna.

details hoyala liyanna tikak wela giya.

kalin messages tikath balala meka liyanne.

message eka liyaddi wena wadakata yanna una.

parcel eka awath danma ganna ona behetha athule na wena dose ekak gedara na.

pennana reference eka SIMB-2VKCLXF.


In [20]:
# Inspect ride/trip issues versus safety incidents
category_pairs = [
    ("ride_trip_issue", "safety_conduct"),
]

for category_a, category_b in category_pairs:
    print(f"\n{'=' * 70}")
    print(f"{category_a.upper()} vs {category_b.upper()}")
    print("=" * 70)

    for category in [category_a, category_b]:
        samples = train_df[train_df["category"] == category].sample(
            3, random_state=42
        )

        print(f"\n--- {category} ---")

        for i, (_, row) in enumerate(samples.iterrows(), 1):
            print(f"\nSample {i}:")
            print(row["text"][:500])


RIDE_TRIP_ISSUE vs SAFETY_CONDUCT

--- ride_trip_issue ---

Sample 1:
agent: thank you for calling RideEat how may I assist
customer: I have been trying to sort this out.
agent: please explain the issue
customer: The driver went to a different street for pickup and blamed me. I checked that this is the support address shown inside the app. The reference shown to me is SIMB-2XQJDMA. I normally use the service around Colombo. Can somebody check the record?
agent: thank you I have noted the request

Sample 2:
Dear RideEat support,

This has been difficult to explain in the app.
The driver ended the trip before reaching my chosen destination.
I usually keep the booking record until everything has been sorted out.
I tried to explain the sequence to a family member before putting it in writing.
I am giving as much background as I can remember so the record can be located.
If you need another detail about the booking, please let me know which one.
I normally use the service around Kandy.
I s

In [21]:
# Inspect app/technical issues versus account and promotion issues
category_pairs = [
    ("app_technical", "account_promo"),
]

for category_a, category_b in category_pairs:
    print(f"\n{'=' * 70}")
    print(f"{category_a.upper()} vs {category_b.upper()}")
    print("=" * 70)

    for category in [category_a, category_b]:
        samples = train_df[train_df["category"] == category].sample(
            3, random_state=42
        )

        print(f"\n--- {category} ---")

        for i, (_, row) in enumerate(samples.iterrows(), 1):
            print(f"\nSample {i}:")
            print(row["text"][:500])


APP_TECHNICAL vs ACCOUNT_PROMO

--- app_technical ---

Sample 1:
uh hello නිලධාරියා: RideEat අමතනවාට ස්තුතියි විස්තර කියන්න
පාරිභෝගිකයා: මගේ ඉල්ලීම බලන්න මම පණිවිඩය ලියන අතරේ වෙන වැඩකට යන්න වුණා
නිලධාරියා: තව විස්තර කියන්න
පාරිභෝගිකයා: මම කලින් පණිවිඩ ටිකත් බලලා මේක ලියන්නේ ඇතුළු වුණාට පස්සේ සිතියම අරිද්දී යෙදුම වැහෙනවා මම මීගමුව පැත්තේ හිටියේ තව විස්තර ඕන නම් කියන්න

Sample 2:
தயவுசெய்து இதைப் பாருங்கள். முந்தைய செய்திகளையும் பார்த்துவிட்டுத்தான் எழுதுகிறேன். உள்நுழைந்த பிறகு வரைபடத்தைத் திறந்தால் செயலி மூடுகிறது. இதுபற்றி முதல் தடவையாகச் செய்தி அனுப்புகிறேன். நான் முன்பே பதிவு செய்திருந்தேன். நான் தொலைபேசியில் பார்த்தேன். இதுபற்றி விசாரித்துச் சொல்லுங்கள்.

Sample 3:
ஒரு உதவி வேணும் வீட்டில் கேட்ட பிறகு விவரங்களை மீண்டும் படித்தேன் நினைவில் உள்ளபடி முதலில் தொலைபேசியில் உறுதிப்படுத்தலைப் பார்த்தேன் உள்நுழைந்த உள்நுழைந்த பிறகு வரைபடத்தைத் திறந்தால் செயலி மூடுகிறது நான் கொழும்பு பகுதியில் இருந்தேன் நான் தொலைபேசியில் பார்த்தேன் குறிப்பு எண்ணைக் கண்டுபிடிக்கச் சிறிது நேரம் எடுத்தது அடுத்

# Text EDA Findings and Preprocessing Decisions

## Overview

This notebook analyzed the textual characteristics of the RideEat multilingual customer support ticket dataset before baseline model development.

The analysis focused on:

* Multilingual and transliterated text
* Text length and vocabulary
* Unicode and character usage
* Structural patterns
* Reference IDs and numbers
* Category-specific vocabulary
* Word-level and character-level TF-IDF features
* Duplicate and template leakage
* Potentially confusing category boundaries
* Initial preprocessing decisions

---

## Step 1: Load the Dataset

Training and validation datasets were loaded from `data/raw/`.

| Dataset    |  Rows | Columns |
| ---------- | ----: | ------: |
| Train      | 4,000 |       8 |
| Validation |   800 |       8 |

The public columns include:

`ticket_id`, `channel`, `subject`, `text`, `language`, `category`, `secondary_category`, `is_urgent`.

The `text` field is the primary textual input.

---

## Step 2: Inspect Text Samples by Language

Random samples were inspected from each language group.

### Key findings

* English tickets can contain conversational structures such as `agent:` and `customer:`.
* Singlish and Tanglish contain transliterated Sinhala/Tamil.
* Sinhala and Tamil contain Unicode scripts.
* Code-switching occurs in some tickets.
* Irregular whitespace, abbreviations, spelling variation, and informal language are present.
* Some reference IDs and other structured information appear inside the text.
* The `language` column represents the dominant language rather than guaranteeing that every character belongs to that language.

### Implication

The preprocessing pipeline should not assume clean, formal, single-language text.

---

## Step 3: Inspect Basic Text Statistics

Text length and word count were calculated for the training data.

Overall:

* Average text length: approximately **318 characters**
* Median text length: approximately **276 characters**
* Average word count: approximately **50 words**
* Median word count: approximately **42 words**

### Key finding

The dataset contains reasonably detailed support tickets rather than extremely short messages.

---

## Step 4: Analyze Text Statistics by Language

| Language | Tickets | Avg. Words | Median Words | Avg. Characters | Median Characters |
| -------- | ------: | ---------: | -----------: | --------------: | ----------------: |
| English  |   1,400 |      72.61 |         73.5 |          414.22 |             427.5 |
| Mixed    |     100 |      43.72 |         42.5 |          291.59 |             265.5 |
| Sinhala  |     800 |      38.11 |         35.0 |          241.91 |             218.0 |
| Singlish |   1,000 |      39.40 |         38.0 |          260.62 |             252.0 |
| Tamil    |     600 |      32.89 |         31.0 |          303.42 |             279.0 |
| Tanglish |     100 |      36.01 |         33.5 |          266.95 |             238.5 |

### Key findings

* English tickets are substantially longer.
* Sinhala, Tamil, Singlish, and Tanglish have shorter average word counts.
* Raw word count is therefore not directly comparable across languages.
* Character-level representations may be particularly useful for multilingual classification.

---

## Step 5: Analyze Common Raw Words by Language

Frequent raw words were inspected separately for each language.

### Key findings

* English contains common conversational words such as `the`, `i`, `to`, `please`, and `help`.
* Singlish contains frequent transliterated Sinhala expressions such as `eka`, `meka`, `mokakda`, and `gana`.
* Tanglish contains transliterated Tamil expressions such as `idha`, `naan`, `enna`, and `solla`.
* Sinhala and Tamil contain their respective Unicode scripts.
* Speaker labels such as `agent:` and `customer:` occur across multiple language groups.
* Code-switching is present.

### Implication

A conventional English-only preprocessing pipeline would be inappropriate.

---

## Step 6: Analyze Latin and Non-Latin Characters

The proportion of Latin and non-Latin alphabetic characters was measured by language.

| Language |   Latin | Non-Latin |
| -------- | ------: | --------: |
| English  | 100.00% |     0.00% |
| Singlish | 100.00% |     0.00% |
| Mixed    |  53.62% |    46.38% |
| Tamil    |   3.76% |    96.24% |
| Sinhala  |   4.60% |    95.40% |
| Tanglish | 100.00% |     0.00% |

### Key findings

* Singlish and Tanglish are primarily Latin-script transliterations.
* Sinhala and Tamil are overwhelmingly represented using Unicode scripts.
* Mixed tickets genuinely combine scripts.
* Character-level features can capture useful patterns across both scripts and transliterations.

---

## Step 7: Inspect Structural Text Patterns

The frequency of common structural patterns was measured.

| Pattern      | Tickets | Percentage |
| ------------ | ------: | ---------: |
| `agent:`     |     637 |     15.93% |
| `customer:`  |     637 |     15.93% |
| Reference ID |     272 |      6.80% |

### Key findings

* Speaker labels are relatively common.
* Reference IDs occur in a smaller but meaningful portion of tickets.
* These patterns may contain useful structure but could also introduce noise.

---

## Step 8: Analyze Structural Patterns by Category

Speaker labels and reference IDs were compared across categories.

### Key findings

* Structural patterns occur across many categories.
* No category showed an obvious strong dependency on these patterns.
* `payment_refund` had somewhat more reference IDs than some categories, but the difference was not strong enough to treat reference IDs as predictive by default.
* `spam_irrelevant` contained no observed reference IDs in this analysis.

### Decision

Keep the original text initially and allow model experiments to determine whether removing structural elements improves performance.

---

## Step 9: Inspect Reference ID Examples

Reference IDs such as:

`simb-4pvsfpq`

were inspected.

### Key finding

The reference IDs appear to be randomly generated identifiers rather than meaningful semantic information.

### Decision

Reference IDs will be preserved in the initial baseline but their removal will be tested later.

If removing them improves validation performance, the cleaned version can be considered for the final pipeline.

---

## Step 10: Analyze Common Raw Words by Primary Category

Frequent words were inspected within each primary category.

### Key findings

* Many common conversational words are shared across categories.
* Some category-specific signals are visible.
* Examples include:

  * `driver` in ride-related and safety tickets
  * `code` in account/promotion tickets
  * `map` and `app` in technical tickets
  * `help` and `explain` in general inquiries
* Multilingual and transliterated words also appear in category-specific patterns.

### Implication

Simple keyword matching will not be sufficient, but word-level TF-IDF should still provide useful baseline features.

---

## Step 11: Inspect Word-Level TF-IDF Features

A word-level TF-IDF representation using unigram and bigram features was examined.

Configuration:

* Lowercasing: enabled
* N-grams: 1-2
* Minimum document frequency: 2

### Key findings

* TF-IDF identifies category-related vocabulary.
* Generic conversational vocabulary remains common.
* Category-specific terms such as `driver`, `code`, `map`, `app`, `help`, and `safe` appear in relevant categories.
* Multilingual tokens are also captured.

### Decision

Word-level TF-IDF is a valid baseline representation.

---

## Step 12: Inspect Character-Level TF-IDF Features

A character-level TF-IDF representation was examined using:

* Character analyzer
* N-grams: 3-5
* Minimum document frequency: 3

The resulting vocabulary contained approximately **45,322 character features**.

### Key findings

Character TF-IDF can capture:

* Partial words
* Spelling variations
* Transliteration patterns
* Prefixes and suffixes
* Unicode character sequences
* Cross-word patterns
* Informal text variations

The raw feature list contains noisy patterns involving whitespace and newline characters, which is expected.

### Decision

Character TF-IDF should be a major baseline rather than only an optional experiment.

---

## Step 13: Check Exact Duplicate Text

Exact text repetition was checked within the training dataset.

### Results

* Unique text values: **4,000**
* Repeated texts: **0**
* Maximum repetitions of one text: **1**

### Finding

Every training ticket has unique text.

There is no evidence of exact duplicate tickets within the training dataset.

---

## Step 14: Check Train-Validation Text Overlap

Exact text overlap between training and validation datasets was checked.

### Result

**Train-validation text overlap: 0**

### Finding

No identical ticket text appears in both datasets.

This provides no evidence of exact-text leakage between training and validation.

---

## Step 15: Check Near-Duplicate Text Templates

Text was normalized by:

* Replacing reference IDs with `<REF>`
* Replacing numbers with `<NUM>`
* Normalizing whitespace
* Converting text to lowercase

The resulting normalized templates were compared.

### Results

* Unique normalized templates: **4,000**
* Repeated templates: **0**
* Maximum repetitions: **1**

### Finding

Even after removing obvious variable elements, every training ticket still had a unique normalized pattern.

There is no evidence of simple repeated text templates causing obvious leakage.

---

## Step 16: Inspect Potentially Confusing Category Pairs

Several category pairs were manually inspected to understand their semantic boundaries.

### Payment Refund vs Order Missing/Wrong

**payment_refund**

Focuses on:

* Payments
* Refunds
* Charges
* Wallet/balance issues

**order_missing_wrong**

Focuses on:

* Missing items
* Incorrect order contents
* Incomplete deliveries

A ticket may mention payment information while its actual primary issue belongs to another category.

---

### Delivery Delay vs Order Missing/Wrong

**delivery_delay**

The order has not arrived or the delivery is delayed.

**order_missing_wrong**

The order has arrived, but items are missing or incorrect.

This distinction cannot reliably be made using keywords alone.

---

### Ride Trip Issue vs Safety Conduct

**ride_trip_issue**

Examples include:

* Wrong pickup location
* Route problems
* Destination problems
* Driver ending a trip early

**safety_conduct**

Focuses on:

* Dangerous driving
* Immediate safety concerns
* Serious inappropriate conduct

Safety takes precedence when a genuine safety incident is present.

---

### App Technical vs Account Promo

**app_technical**

Examples include:

* Application crashes
* Technical failures
* App functionality problems

**account_promo**

Examples include:

* OTP/login-code problems
* Account issues
* Promotion-related issues

Some tickets contain multiple issues. Therefore, the model needs to identify the primary intent rather than simply detect the presence of a keyword.

### Overall finding

The dataset contains multi-issue and conversational tickets. Successful classification requires contextual understanding rather than simple keyword rules.

---

## Step 17: Initial Text Preprocessing Decisions

Based on the complete text EDA, the initial modeling pipeline will use the following strategy.

| Feature / preprocessing       | Initial decision                 |
| ----------------------------- | -------------------------------- |
| Unicode Sinhala/Tamil         | Preserve                         |
| Singlish/Tanglish             | Preserve                         |
| Code-switching                | Preserve                         |
| Punctuation                   | Preserve initially               |
| Numbers                       | Preserve initially               |
| `agent:` / `customer:` labels | Preserve initially               |
| Reference IDs                 | Preserve initially; test removal |
| Lowercasing                   | Test experimentally              |
| Stopword removal              | Do not apply initially           |
| Word TF-IDF                   | Include                          |
| Character TF-IDF              | Include                          |
| Subject                       | Test as an additional feature    |
| Language                      | Test as metadata                 |
| Channel                       | Test as metadata                 |

### Reasoning

The dataset contains multilingual, transliterated, informal, and code-switched text. Aggressive normalization could remove useful information.

Therefore, the first experiments should use relatively light preprocessing and allow validation performance to determine whether additional cleaning is beneficial.

---

# Overall Text EDA Conclusions

The text analysis provides several important conclusions for model development:

1. **The dataset is genuinely multilingual.**
   Sinhala, Tamil, English, Singlish, Tanglish, and mixed-language tickets are present.

2. **Transliteration is important.**
   Singlish and Tanglish use Latin characters while representing Sinhala and Tamil language content.

3. **Character-level features are strongly justified.**
   They can capture spelling variation, transliteration, partial words, and multilingual character patterns.

4. **Word-level features remain useful.**
   Category-specific vocabulary exists despite substantial shared conversational language.

5. **The text contains multiple issues in some tickets.**
   Classification requires identifying the primary intent rather than matching isolated keywords.

6. **Safety requires contextual classification.**
   A normal ride problem should not automatically become a safety incident simply because a driver is mentioned.

7. **Aggressive text cleaning should be avoided initially.**
   Unicode, punctuation, numbers, transliteration, and conversational structure may contain useful information.

8. **There is no evidence of obvious text leakage.**
   No exact duplicates, train-validation text overlap, or repeated normalized templates were found.

9. **Subject information is limited.**
   Approximately 70% of subjects are missing, so the main model should rely primarily on `text`.

10. **The baseline experiments should compare representations rather than assume one is best.**

---

# Next Phase: Baseline Modeling

The next notebook, `03_baseline_models.ipynb`, will compare progressively stronger approaches:

1. Word TF-IDF + Logistic Regression
2. Word TF-IDF + Linear SVM
3. Character TF-IDF + Logistic Regression
4. Character TF-IDF + Linear SVM
5. Combined word + character TF-IDF
6. Add `language` and `channel` metadata
7. Compare `text` against `subject + text`
8. Evaluate per-category performance
9. Evaluate per-language performance
10. Analyze confusion matrices and classification errors

These experiments will establish a strong classical baseline before deciding whether a multilingual transformer such as **XLM-R** provides enough additional value to justify its computational cost and complexity.
